In [7]:
%%writefile preprocessing_test.h

#pragma once
#ifndef PREPROCESS_H
#define PREPROCESS_H

void preprocess(char* RefSeq, char* ReadSeq, int ReadLength)
{
    int i, index = 0;

#define ENCODE_BASE(b) \
        ((b) == 'A' || (b) == 'a' ? 0b0001 : \
         (b) == 'C' || (b) == 'c' ? 0b0010 : \
         (b) == 'G' || (b) == 'g' ? 0b0011 : \
         (b) == 'T' || (b) == 't' ? 0b0100 : \
         (b) == 'N' || (b) == 'n' ? 0b0101 : 0x00)

    for (i = 0; i < ReadLength; i += 2) {
        unsigned char base1_r = ENCODE_BASE(ReadSeq[i]);
        unsigned char base1_f = ENCODE_BASE(RefSeq[i]);

        unsigned char base2_r = 0;
        unsigned char base2_f = 0;

        if (i + 1 < ReadLength) {
            base2_r = ENCODE_BASE(ReadSeq[i + 1]);
            base2_f = ENCODE_BASE(RefSeq[i + 1]);
        }
        else {
            base2_r = 0x0F; // Padding for odd length
            base2_f = 0x0F; // Padding for odd length
        }

        // Pack two 4-bit bases into one byte
        ReadSeq[index] = (char)((base1_r << 4) | base2_r);
        RefSeq[index] = (char)((base1_f << 4) | base2_f);
        index++;
    }

#undef ENCODE_BASE
}

#endif // PREPROCESS_H

Overwriting preprocessing_test.h


In [ ]:
%%writefile checkpointfull_test.c

//with preprocessing timer

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <stdint.h>
#include <time.h>
#include "preprocessing_test.h"

static inline uint64_t rdtsc(void) {
    uint32_t lo, hi;
    __asm__ volatile ("rdtsc" : "=a"(lo), "=d"(hi));
    return ((uint64_t)hi << 32) | lo;
}


typedef struct {
    uint64_t start_cycles;
    uint64_t total_cycles;
    uint64_t call_count;
} Timer;

typedef struct {
    Timer preprocess_timer;
    Timer main_diagonal_timer;
    Timer right_diagonal_timer;
    Timer left_diagonal_timer;
    Timer snake_total_timer;
    Timer accepted_cycles_timer;
    Timer rejected_cycles_timer;
} TimingData;

TimingData timing_data = {0};

static double cycles_to_seconds(uint64_t cycles) {
    static double cpu_freq_ghz = 2.4;
    return cycles / (cpu_freq_ghz * 1e9);
}

void print_timing_breakdown(int n_reads, int n_accepted, long long total_bases) {
    printf("\n=== ASM INTERNAL TIMING BREAKDOWN ===\n");
    printf("%-28s %12s %12s %15s\n", "Component", "Total (s)", "Calls", "Avg Cycles");
    printf("-------------------------------------------------------------------------------\n");

    #define PRINT_TIMER(name, t) \
        printf("%-28s %12.6f %12lu %15.0f\n", \
               name, \
               cycles_to_seconds(t.total_cycles), \
               t.call_count, \
               t.call_count > 0 ? (double)t.total_cycles / t.call_count : 0.0)

    PRINT_TIMER("Preprocessing (C)",       timing_data.preprocess_timer);
    PRINT_TIMER("Total SneakySnake",       timing_data.snake_total_timer);
    PRINT_TIMER("  Main Diagonal",         timing_data.main_diagonal_timer);
    PRINT_TIMER("  Right (Upper) Diag",    timing_data.right_diagonal_timer);
    PRINT_TIMER("  Left (Lower) Diag",     timing_data.left_diagonal_timer);

    uint64_t components = timing_data.main_diagonal_timer.total_cycles +
                          timing_data.right_diagonal_timer.total_cycles +
                          timing_data.left_diagonal_timer.total_cycles;

    if (timing_data.snake_total_timer.total_cycles > components) {
        uint64_t overhead = timing_data.snake_total_timer.total_cycles - components;
        printf("%-28s %12.6f %12s %15s\n",
               "  Other/Overhead", cycles_to_seconds(overhead), "-", "-");
    }
    printf("-------------------------------------------------------------------------------\n");
    PRINT_TIMER("  Accepted reads",        timing_data.accepted_cycles_timer);
    PRINT_TIMER("  Rejected reads",        timing_data.rejected_cycles_timer);
    printf("-------------------------------------------------------------------------------\n");

    #undef PRINT_TIMER

    // ---- Throughput metrics ----
    int n_rejected = n_reads - n_accepted;
    uint64_t tot     = timing_data.snake_total_timer.total_cycles;
    uint64_t acc_cyc = timing_data.accepted_cycles_timer.total_cycles;
    uint64_t rej_cyc = timing_data.rejected_cycles_timer.total_cycles;

    printf("\n  -- Throughput --\n");
    if (n_reads > 0)
        printf("  Cycles / read      : %.1f\n", (double)tot / n_reads);

    if (n_accepted > 0)
        printf("  Cycles / accepted  : %.1f\n", (double)acc_cyc / n_accepted);
    else
        printf("  Cycles / accepted  : n/a (0 accepted)\n");

    if (n_rejected > 0)
        printf("  Cycles / rejected  : %.1f\n", (double)rej_cyc / n_rejected);
    else
        printf("  Cycles / rejected  : n/a (0 rejected)\n");

    if (total_bases > 0)
        printf("  Cycles / base      : %.3f\n", (double)tot / (double)total_bases);

    // Preprocessing throughput
    uint64_t prep_cyc = timing_data.preprocess_timer.total_cycles;
    if (n_reads > 0)
        printf("  Preprocess cyc/read: %.1f\n", (double)prep_cyc / n_reads);
    if (total_bases > 0)
        printf("  Preprocess cyc/base: %.3f\n", (double)prep_cyc / (double)total_bases);

    printf("\n");
}

void reset_asm_timers(void) {
    memset(&timing_data, 0, sizeof(TimingData));
}


static inline uint8_t decode_nibble(uint8_t nibble) {
    switch (nibble & 0x0F) {
        case 0b0001: return 'A';
        case 0b0010: return 'C';
        case 0b0011: return 'G';
        case 0b0100: return 'T';
        case 0b0101: return 'N';
        default:     return '?';
    }
}

void print_sequence_debug(const char *label,
                          const uint8_t *ref_enc, const uint8_t *read_enc,
                          int len, int kmer_size, int threshold, int show_len)
{
    if (show_len > len) show_len = len;
    printf("\nSEQUENCE DEBUG: %s\n", label);
    printf("Length=%d  KmerSize=%d  Threshold=%d  Showing first %d bases\n",
           len, kmer_size, threshold, show_len);

    int total_mm = 0;
    for (int start = 0; start < show_len; start += 64) {
        int end = start + 64;
        if (end > show_len) end = show_len;

        printf("\n  Pos  [%4d - %4d]\n", start, end - 1);
        printf("  Kmer  ");
        for (int i = start; i < end; i++)
            printf("%c", (kmer_size > 0 && i % kmer_size == 0) ? '|' : '-');
        printf("\n  REF   ");
        for (int i = start; i < end; i++) {
            int byte_idx = i / 2;
            int is_lower = i % 2;
            uint8_t nibble = is_lower ? (ref_enc[byte_idx] & 0x0F) : (ref_enc[byte_idx] >> 4);
            printf("%c", decode_nibble(nibble));
        }
        printf("\n  READ  ");
        for (int i = start; i < end; i++) {
            int byte_idx = i / 2;
            int is_lower = i % 2;
            uint8_t nibble = is_lower ? (read_enc[byte_idx] & 0x0F) : (read_enc[byte_idx] >> 4);
            printf("%c", decode_nibble(nibble));
        }
        printf("\n  MATCH ");
        for (int i = start; i < end; i++) {
            int byte_idx = i / 2;
            int is_lower = i % 2;
            uint8_t ref_nib  = is_lower ? (ref_enc[byte_idx] & 0x0F) : (ref_enc[byte_idx] >> 4);
            uint8_t read_nib = is_lower ? (read_enc[byte_idx] & 0x0F) : (read_enc[byte_idx] >> 4);
            printf("%c", (ref_nib == read_nib) ? '1' : '0');
        }
        printf("\n  DIFF  ");
        int mm = 0;
        for (int i = start; i < end; i++) {
            int byte_idx = i / 2;
            int is_lower = i % 2;
            uint8_t ref_nib  = is_lower ? (ref_enc[byte_idx] & 0x0F) : (ref_enc[byte_idx] >> 4);
            uint8_t read_nib = is_lower ? (read_enc[byte_idx] & 0x0F) : (read_enc[byte_idx] >> 4);
            if (ref_nib != read_nib) { printf("^"); mm++; } else printf(" ");
        }
        printf("  (%d mismatches)\n", mm);
        total_mm += mm;
    }
    printf("\n  SUMMARY: bases=%d  mismatches=%d  match=%.1f%%\n\n",
           show_len, total_mm, 100.0 * (show_len - total_mm) / show_len);
}


static inline uint8_t get_nibble(uint8_t *seq, int base_index) {
    int byte_idx = base_index / 2;
    int is_lower = base_index % 2;
    return is_lower ? (seq[byte_idx] & 0x0F) : ((seq[byte_idx] >> 4) & 0x0F);
}

int SneakySnake_C(int ReadLength, uint8_t *RefSeq, uint8_t *ReadSeq,
                  int EditThreshold, int IterationNo, int KmerSize)
{
    int Edits = 0;

    if (KmerSize == 0 || KmerSize >= ReadLength)
        KmerSize = ReadLength;

    int NumKmers = ReadLength / KmerSize;

    for (int K = 0; K < NumKmers; K++) {
        int KmerStart = K * KmerSize;
        int KmerEnd   = (K < NumKmers - 1) ? (K + 1) * KmerSize : ReadLength;

        int index    = KmerStart;
        int roundsNo = 1;

        while (index < KmerEnd) {
            if (roundsNo > IterationNo)
                goto LOOP;

            if (Edits > EditThreshold)
                return 0;

            // Main diagonal
            int GlobalCount = 0;
            int count = 0;
            for (int n = index; n < KmerEnd; n++) {
                if (get_nibble(ReadSeq, n) != get_nibble(RefSeq, n)) break;
                count++;
            }
            GlobalCount = count;

            if (GlobalCount == (KmerEnd - KmerStart))
                goto LOOP;

            // Off-diagonals
            for (int e = 1; e <= EditThreshold; e++) {
                // Right diagonal
                count = 0;
                for (int n = index; n < KmerEnd; n++) {
                    if (n < e) break;
                    if (get_nibble(ReadSeq, n - e) != get_nibble(RefSeq, n)) break;
                    count++;
                }
                if (count > GlobalCount) GlobalCount = count;
                if (count == (KmerEnd - KmerStart)) goto LOOP;

                // Left diagonal
                count = 0;
                for (int n = index; n < KmerEnd; n++) {
                    if (n > ReadLength - e - 1) break;
                    if (get_nibble(ReadSeq, n + e) != get_nibble(RefSeq, n)) break;
                    count++;
                }
                if (count > GlobalCount) GlobalCount = count;
                if (count == (KmerEnd - KmerStart)) goto LOOP;
            }

            index += GlobalCount;

            if (index < KmerEnd) {
                Edits++;
                index++;
            }

            roundsNo++;

            if (Edits > EditThreshold)
                return 0;
        }

        LOOP:
        if (Edits > EditThreshold)
            return 0;
    }

    return (Edits <= EditThreshold) ? 1 : 0;
}


extern uint64_t SneakySnake(uint64_t ReadLength, uint8_t *RefSeq,
                             uint8_t *ReadSeq, uint64_t EditThreshold,
                             uint64_t IterationNo, uint64_t KmerSize);

extern uint64_t current_position;
extern uint64_t current_edits;
extern uint64_t mismatch_count;
extern uint64_t safety_counter;


int main(int argc, char *argv[]) {
    if (argc < 5) {
        printf("Usage: %s <file> <threshold> <seq_len> <kmer_size> [limit] [debug_limit]\n", argv[0]);
        printf("  kmer_size: Size of kmers in bases (0 = no kmer, process whole sequence)\n");
        return 1;
    }

    const char *filename   = argv[1];
    int threshold          = atoi(argv[2]);
    int fixed_len          = atoi(argv[3]);
    int kmer_size          = atoi(argv[4]);
    int limit              = (argc >= 6) ? atoi(argv[5]) : 30000;
    int debug_limit        = (argc >= 7) ? atoi(argv[6]) : 5;

    if (kmer_size < 0) {
        printf("Error: kmer_size must be >= 0\n");
        return 1;
    }
    if (kmer_size > 0 && kmer_size > fixed_len) {
        printf("Warning: kmer_size (%d) > seq_len (%d), will process as single segment\n",
               kmer_size, fixed_len);
    }

    FILE *file = fopen(filename, "r");
    if (!file) {
        printf("Error: Cannot open file %s\n", filename);
        return 1;
    }
    setvbuf(file, NULL, _IOFBF, 8 * 1024 * 1024);

    printf("Loading sequences (Fixed Len=%d, KmerSize=%d, Limit=%d)...\n",
           fixed_len, kmer_size, limit);

    int      *lengths    = malloc(limit * sizeof(int));
    int      *edit_dists = malloc(limit * sizeof(int));
    uint8_t **read_enc   = malloc(limit * sizeof(uint8_t *));
    uint8_t **ref_enc    = malloc(limit * sizeof(uint8_t *));

    if (!lengths || !edit_dists || !read_enc || !ref_enc) {
        perror("malloc");
        return 1;
    }

    size_t stride    = (size_t)(fixed_len + 64);
    uint8_t *read_pool = aligned_alloc(64, (size_t)limit * stride);
    uint8_t *ref_pool  = aligned_alloc(64, (size_t)limit * stride);
    if (!read_pool || !ref_pool) { perror("aligned_alloc"); return 1; }

    reset_asm_timers();

    char  *line = NULL;
    size_t cap  = 0;
    int    count = 0;

    while (getline(&line, &cap, file) != -1 && count < limit) {
        char *read_seq = line;
        char *ref_seq  = strpbrk(line, "\t ");
        if (!ref_seq) continue;
        *ref_seq = '\0';
        ref_seq++;
        while (*ref_seq == ' ' || *ref_seq == '\t') ref_seq++;
        ref_seq[strcspn(ref_seq, "\r\n")] = '\0';

        int actual_len = (int)strlen(read_seq);
        if (actual_len < fixed_len) continue;

        int len = fixed_len;
        read_seq[len] = '\0';
        ref_seq[len]  = '\0';

        lengths[count]    = len;
        edit_dists[count] = 0;  

        read_enc[count] = read_pool + (size_t)count * stride;
        ref_enc[count]  = ref_pool  + (size_t)count * stride;
        memset(read_enc[count], 0, stride);
        memset(ref_enc[count],  0, stride);

        memcpy(read_enc[count], read_seq, len);
        memcpy(ref_enc[count],  ref_seq,  len);

        uint64_t prep_start = rdtsc();
        preprocess(ref_enc[count], read_enc[count], len);
        uint64_t prep_end = rdtsc();

        timing_data.preprocess_timer.total_cycles += (prep_end - prep_start);
        timing_data.preprocess_timer.call_count++;

        count++;
        if (count % 500000 == 0) {
            printf("  Loaded %d sequences...\r", count);
            fflush(stdout);
        }
    }

    fclose(file);
    free(line);

    printf("\nLoaded %d sequences total.\n", count);

    long long total_bases = 0;
    for (int i = 0; i < count; i++) total_bases += lengths[i];


    printf("\n=== DEBUGGING FIRST %d SEQUENCES ===\n", debug_limit);

    for (int i = 0; i < debug_limit && i < count; i++) {
        if (lengths[i] > 5000) {
            printf("\n[%d] Len=%d (skipping debug for long sequence)\n", i, lengths[i]);
            continue;
        }

        printf("\n--- Pair #%d  Threshold=%d ---\n", i, threshold);

        char lbl[64];
        snprintf(lbl, sizeof(lbl), "Pair #%d", i);
        int show_len = (lengths[i] < 128) ? lengths[i] : 128;
        print_sequence_debug(lbl, ref_enc[i], read_enc[i],
                             lengths[i], kmer_size, threshold, show_len);

        int c_result = SneakySnake_C(lengths[i], ref_enc[i], read_enc[i],
                                     threshold, lengths[i] * 2, kmer_size);

        current_position = 0;
        current_edits    = 0;
        mismatch_count   = 0;
        safety_counter   = 0;

        int asm_result = (int)SneakySnake(lengths[i], ref_enc[i], read_enc[i],
                                          threshold, lengths[i] * 2, kmer_size);

        printf("\n[%d] Len=%d  Kmers=%d\n",
               i, lengths[i],
               (kmer_size > 0 && kmer_size < lengths[i]) ? (lengths[i] / kmer_size) : 1);
        printf("    C:    %s\n", c_result ? "ACCEPT" : "REJECT");
        printf("    ASM:  %s (edits=%lu, pos=%lu, safety=%lu)\n",
               asm_result ? "ACCEPT" : "REJECT",
               current_edits, current_position, safety_counter);

        if (c_result != asm_result)
            printf("    *** C AND ASM DISAGREE! ***\n");
        else
            printf("    Agreement: OK\n");
    }

    printf("\n=== RUNNING FULL BENCHMARK ===\n");

    int c_matches   = 0;
    int asm_matches = 0;

    //C benchmark
    clock_t c_start = clock();
    for (int i = 0; i < count; i++) {
        if (SneakySnake_C(lengths[i], ref_enc[i], read_enc[i],
                          threshold, lengths[i] * 2, kmer_size))
            c_matches++;
    }
    clock_t c_end  = clock();
    double  c_time = (double)(c_end - c_start) / CLOCKS_PER_SEC;

    timing_data.main_diagonal_timer    = (Timer){0};
    timing_data.right_diagonal_timer   = (Timer){0};
    timing_data.left_diagonal_timer    = (Timer){0};
    timing_data.snake_total_timer      = (Timer){0};
    timing_data.accepted_cycles_timer  = (Timer){0};
    timing_data.rejected_cycles_timer  = (Timer){0};

    //ASM benchmark
    clock_t asm_start = clock();
    for (int i = 0; i < count; i++) {
        current_position = 0;
        current_edits    = 0;
        mismatch_count   = 0;
        safety_counter   = 0;

        // time for accepted/rejected split
        uint64_t t0 = rdtsc();
        int result = (int)SneakySnake(lengths[i], ref_enc[i], read_enc[i],
                                      threshold, lengths[i] * 2, kmer_size);
        uint64_t t1 = rdtsc();
        uint64_t elapsed = t1 - t0;

        if (result) {
            asm_matches++;
            timing_data.accepted_cycles_timer.total_cycles += elapsed;
            timing_data.accepted_cycles_timer.call_count++;
        } else {
            timing_data.rejected_cycles_timer.total_cycles += elapsed;
            timing_data.rejected_cycles_timer.call_count++;
        }

        // Also accumulate into snake_total
        timing_data.snake_total_timer.total_cycles += elapsed;
        timing_data.snake_total_timer.call_count++;
    }
    clock_t asm_end  = clock();
    double  asm_time = (double)(asm_end - asm_start) / CLOCKS_PER_SEC;

   
    printf("\n==================================================\n");
    printf("                  BENCHMARK SUMMARY               \n");
    printf("==================================================\n");
    printf("Total Pairs              : %d\n", count);
    printf("Fixed Length             : %d bases\n", fixed_len);
    printf("Kmer Size                : %d bases\n", kmer_size);
    printf("Threshold                : %d\n", threshold);
    printf("\n");
    printf("C Implementation:\n");
    printf("  Accepted               : %d\n", c_matches);
    printf("  Time                   : %.4f seconds\n", c_time);
    printf("\n");
    printf("ASM Implementation:\n");
    printf("  Accepted               : %d\n", asm_matches);
    printf("  Time                   : %.4f seconds\n", asm_time);
    if (asm_time > 0)
        printf("  Speedup vs C           : %.2fx\n", c_time / asm_time);

    print_timing_breakdown(count, asm_matches, total_bases);

    if (c_matches != asm_matches) {
        printf("\n==================================================\n");
        printf("WARNING: C and ASM implementations disagree!\n");
        printf("C accepted %d, ASM accepted %d (diff = %d)\n",
               c_matches, asm_matches, abs(c_matches - asm_matches));
        printf("==================================================\n");
    }

    free(read_pool);
    free(ref_pool);
    free(read_enc);
    free(ref_enc);
    free(lengths);
    free(edit_dists);

    return 0;
}

Overwriting checkpointfull_test.c


In [10]:
%%writefile checkpointavxfull_test.asm

default rel
bits 64

%define VCMP_NEQ 4

%define OFF_MAIN_DIAG  24
%define OFF_RIGHT_DIAG 48
%define OFF_LEFT_DIAG  72
%define OFF_TOTAL      96
%define OFF_ACCEPTED   120
%define OFF_REJECTED   144

; Macro to start a timer
%macro START_TIMER 1
    push    rax
    push    rdx
    rdtsc                   
    shl     rdx, 32
    or      rax, rdx        
    mov     %1, rax         
    pop     rdx
    pop     rax
%endmacro

%macro STOP_TIMER 2
    push    rax
    push    rdx
    push    rcx             
    
    rdtsc                   
    shl     rdx, 32
    or      rax, rdx
    
    sub     rax, %1         
    
    lea     rcx, [timing_data + %2]
    
    add     [rcx + 8], rax
    
    inc     qword [rcx + 16]
    
    pop     rcx
    pop     rdx
    pop     rax
%endmacro

%macro ACCUM_OUTCOME 1
    push    rax
    push    rdx
    push    rcx
    rdtsc
    shl     rdx, 32
    or      rax, rdx
    sub     rax, [rbp-64]              ; elapsed since function entry
    lea     rcx, [timing_data + %1]
    add     [rcx + 8], rax             ; total_cycles += elapsed
    inc     qword [rcx + 16]           ; call_count++
    pop     rcx
    pop     rdx
    pop     rax
%endmacro
    
section .data
global SneakySnake
global current_position
global current_edits
global mismatch_count
global safety_counter

extern timing_data

align 64
current_position dq 0
current_edits dq 0
mismatch_count dq 0
safety_counter dq 0

section .text
global SneakySnake

; Registers:
;   r13 = ReadLength
;   r12 = RefSeq
;   r11 = ReadSeq
;   r10 = EditThreshold
;   rbx = KmerEnd (register-based, eliminating memory loads)
;   r15 = index/KmerStart
;   r14 = Edits
;
; Stack:
;    8 = KmerSize
;   16 = roundsNo
;   24 = IterationNo
;   32 = main_timer
;   40 = GlobalCount storage
;   48 = upper_timer
;   56 = lower_timer
;   64 = total_timer  

SneakySnake:
    push    rbp
    mov     rbp, rsp
    push    rbx
    push    r12
    push    r13
    push    r14
    push    r15
    sub     rsp, 72             

    START_TIMER [rbp-64]        

   
    xor     rax, rax
    mov     [current_position], rax
    mov     [current_edits], rax
    mov     [mismatch_count], rax
    mov     [safety_counter], rax

    mov     r13, rdi              ; ReadLength
    mov     r12, rsi              ; RefSeq
    mov     r11, rdx              ; ReadSeq
    mov     r10, rcx              ; EditThreshold
    mov     [rbp-24], r8          ; IterationNo
    mov     [rbp-8], r9           ; KmerSize


    mov     r8d, 0x0F0F0F0F
    vpbroadcastd zmm6, r8d

    mov     rbx, r9               ; KmerEnd 
    cmp     rbx, r13
    cmova   rbx, r13              ; first KmerEnd = min(KmerSize, ReadLength)

    xor     r15, r15              ; index, KmerStart = 0
    xor     r14, r14              ; Edits = 0
    mov     qword [rbp-16], 1     ; roundsNo = 1

.while_loop:
    cmp     r15, rbx              ; index < KmerEnd?
    jae     .kmer_next

    inc     qword [safety_counter] ; for iterationNo 

    cmp     r14, r10 ; Edits > EditThreshold?
    jg      .reject

    START_TIMER [rbp-32]

    xor     r9, r9                ; GlobalCount = 0
    
    mov     rcx, rbx              ; calculate remaining kmersize
    sub     rcx, r15              ; rcx = KmerEnd - index

; ------------- MAIN DIAGONAL -------------------------------------
.main_diag:
    lea     rax, [r15 + r9]       ; position = index + match_count

    cmp     rax, rbx              ; vs KmerEnd
    jae     .main_done

    test    al, 1
    jnz     .main_scalar          ; check if odd

    mov     rdx, rbx
    sub     rdx, rax
    cmp     rdx, 128
    jb      .main_scalar          ; check if <128 nibbles

    mov     rsi, rax
    shr     rsi, 1                ; calculate byte offset

    ; prefetch
    prefetcht0 [r11 + rsi + 256]
    prefetcht0 [r12 + rsi + 256]

.main_vector_loop:
    vmovdqu8 zmm0, [r11 + rsi]
    vmovdqu8 zmm1, [r12 + rsi]

    ; get high nibbles (even)
    vmovdqa64 zmm2, zmm0
    vmovdqa64 zmm3, zmm1
    vpsrld zmm2, zmm2, 4
    vpsrld zmm3, zmm3, 4
    vpandd zmm2, zmm2, zmm6
    vpandd zmm3, zmm3, zmm6

    ; get low nibbles (odd)
    vmovdqa64 zmm4, zmm0
    vmovdqa64 zmm5, zmm1
    vpandd zmm4, zmm4, zmm6
    vpandd zmm5, zmm5, zmm6

    ; comparison
    vpcmpub k3, zmm2, zmm3, VCMP_NEQ
    vpcmpub k4, zmm4, zmm5, VCMP_NEQ

    ; check if all matched
    korq k5, k3, k4
    ktestq k5, k5
    jnz .main_mismatch

    add r9, 128
    add rsi, 64
    
    ; check if can do another vector iteration
    sub rdx, 128
    cmp rdx, 128
    jae .main_vector_loop
    
    ; tail handling
    test rdx, rdx
    jz .main_done
    jmp .main_diag

.main_mismatch:
    ; find first mismatch
    kmovq r8, k3
    kmovq rdi, k4

    tzcnt r8, r8
    shl r8, 1                     ; even position

    tzcnt rdi, rdi
    lea rdi, [rdi*2+1]            ; odd position

    cmp r8, rdi
    cmovbe rdx, r8
    cmova rdx, rdi

    add r9, rdx
    jmp .main_done

.main_scalar:
    lea rax, [r15 + r9]

    cmp rax, rbx                  ; vs KmerEnd
    jae .main_done

    mov rdx, rax
    shr rdx, 1
    movzx edi, byte [r11 + rdx]
    movzx esi, byte [r12 + rdx]

    test al, 1
    jz .main_even
    and dil, 0x0F
    and sil, 0x0F
    jmp .main_cmp

.main_even:
    shr dil, 4
    shr sil, 4

.main_cmp:
    cmp dil, sil
    jne .main_done

    inc r9
    jmp .main_diag

.main_done:
    STOP_TIMER [rbp-32], OFF_MAIN_DIAG

    ; check if full kmer matched
    cmp     r9, rcx               ; GlobalCount vs kmer_size_remaining
    jae     .kmer_next

    mov     qword [rbp-40], r9    ; update GlobalCount
    mov     r8, 1                 ; shift = 1

.shift_loop:
    cmp     r8, r10               
    ja      .shift_done

;------------------------- UPPER DIAGONALS ---------------------------
    cmp     r15, r8               ; index < shift ?
    jb      .check_lower

    START_TIMER [rbp-48]

    xor     rax, rax              ; count = 0
    mov     rdi, r15              ; ref_pos = index
    mov     rsi, r15
    sub     rsi, r8               ; read_pos = index - shift

    mov     rdx, rbx
    sub     rdx, rdi              ; KmerEnd - ref_pos
    mov     r9, r13
    sub     r9, rsi               ; ReadLength - read_pos
    cmp     rdx, r9
    cmovg   rdx, r9               ; rdx = min(remaining_kmer, remaining_read)

.upper_diag:
    ;check if odd (or not aligned) and <128 nibbles
    test    dil, 1
    jnz     .upper_scalar
    test    sil, 1
    jnz     .upper_scalar
    cmp     rdx, 128
    jb      .upper_scalar

    ; calculate byte offsets
    mov     rcx, rsi
    shr     rcx, 1
    vmovdqu8 zmm0, [r11 + rcx]

    mov     rcx, rdi
    shr     rcx, 1
    vmovdqu8 zmm1, [r12 + rcx]

    ; extract high nibbles (even)
    vmovdqa64 zmm2, zmm0
    vmovdqa64 zmm3, zmm1
    vpsrld zmm2, zmm2, 4
    vpsrld zmm3, zmm3, 4
    vpandd zmm2, zmm2, zmm6
    vpandd zmm3, zmm3, zmm6

    ; extract low nibbles (odd)
    vmovdqa64 zmm4, zmm0
    vmovdqa64 zmm5, zmm1
    vpandd zmm4, zmm4, zmm6
    vpandd zmm5, zmm5, zmm6

    vpcmpub k3, zmm2, zmm3, VCMP_NEQ
    vpcmpub k4, zmm4, zmm5, VCMP_NEQ

    korq k5, k3, k4
    ktestq k5, k5
    jnz .upper_mismatch

    add rax, 128
    add rdi, 128      
    add rsi, 128
    sub rdx, 128
    
    cmp rdx, 128
    jae .upper_diag

    ;scalar for remainder
    test rdx, rdx
    jz .upper_done
    jmp .upper_diag

.upper_mismatch:
    kmovq rcx, k3
    kmovq r9, k4

    tzcnt rcx, rcx
    shl rcx, 1

    tzcnt r9, r9
    lea r9, [r9*2+1]

    cmp rcx, r9
    cmovbe rcx, rcx
    cmova rcx, r9

    add rax, rcx
    jmp .upper_done

.upper_scalar:
    test    rdx, rdx
    jz      .upper_done

    ; compare nibbles
    mov     rcx, rsi
    shr     rcx, 1
    movzx   r9d, byte [r11 + rcx]
    test    sil, 1
    jz      .upper_read_even
    and     r9d, 0x0F
    jmp     .upper_read_done

.upper_read_even:
    shr     r9d, 4

.upper_read_done:
    mov     rcx, rdi
    shr     rcx, 1
    movzx   ecx, byte [r12 + rcx]
    test    dil, 1
    jz      .upper_ref_even
    and     ecx, 0x0F
    jmp     .upper_ref_done

.upper_ref_even:
    shr     ecx, 4

.upper_ref_done:
    cmp     r9d, ecx
    jne     .upper_done
    inc     rax
    inc     rdi                  
    inc     rsi
    dec     rdx
    jmp     .upper_diag

.upper_done:
    STOP_TIMER [rbp-48], OFF_RIGHT_DIAG

    mov     rcx, rbx
    sub     rcx, r15
    cmp     rax, rcx
    jae     .kmer_next

    cmp     rax, [rbp-40]
    jbe     .check_lower

    mov     [rbp-40], rax

; ----------------------- LOWER DIAGONALS ----------------------------------------
.check_lower:
    lea     rsi, [r15 + r8]       ; read_pos = index + shift

    cmp     rsi, r13              ; read_pos < ReadLength?
    jae     .next_shift

    START_TIMER [rbp-56]          

    xor     rax, rax              ; count = 0
    mov     rdi, r15              ; ref_pos = index

    
    mov     rdx, rbx
    sub     rdx, rdi              ; KmerEnd - ref_pos
    mov     r9, r13
    sub     r9, rsi               ; ReadLength - read_pos
    cmp     rdx, r9
    cmovg   rdx, r9               ; rdx = min(remaining_kmer, remaining_read)

.lower_diag:
    ;check if odd (or not aligned) and <128 nibbles
    test    dil, 1
    jnz     .lower_scalar
    test    sil, 1
    jnz     .lower_scalar
    cmp     rdx, 128
    jb      .lower_scalar

    mov     rcx, rsi
    shr     rcx, 1
    vmovdqu8 zmm0, [r11 + rcx]

    mov     rcx, rdi
    shr     rcx, 1
    vmovdqu8 zmm1, [r12 + rcx]

    ; extract high nibbles (even)
    vmovdqa64 zmm2, zmm0
    vmovdqa64 zmm3, zmm1
    vpsrld zmm2, zmm2, 4
    vpsrld zmm3, zmm3, 4
    vpandd zmm2, zmm2, zmm6
    vpandd zmm3, zmm3, zmm6

    ; extract low nibbles (odd)
    vmovdqa64 zmm4, zmm0
    vmovdqa64 zmm5, zmm1
    vpandd zmm4, zmm4, zmm6
    vpandd zmm5, zmm5, zmm6

    vpcmpub k3, zmm2, zmm3, VCMP_NEQ
    vpcmpub k4, zmm4, zmm5, VCMP_NEQ

    korq k5, k3, k4
    ktestq k5, k5
    jnz .lower_mismatch

    add rax, 128
    add rdi, 128              
    add rsi, 128
    sub rdx, 128
    
    cmp rdx, 128
    jae .lower_diag
    
    ;scalar for remainder
    test rdx, rdx
    jz .lower_done
    jmp .lower_diag

.lower_mismatch:
    kmovq rcx, k3
    kmovq r9, k4

    tzcnt rcx, rcx
    shl rcx, 1

    tzcnt r9, r9
    lea r9, [r9*2+1]

    cmp rcx, r9
    cmovbe rcx, rcx
    cmova rcx, r9

    add rax, rcx
    jmp .lower_done

.lower_scalar:
    test    rdx, rdx
    jz      .lower_done

    ; compare nibbles
    mov     rcx, rsi
    shr     rcx, 1
    movzx   r9d, byte [r11 + rcx]
    test    sil, 1
    jz      .lower_read_even
    and     r9d, 0x0F
    jmp     .lower_read_done

.lower_read_even:
    shr     r9d, 4

.lower_read_done:
    mov     rcx, rdi
    shr     rcx, 1
    movzx   ecx, byte [r12 + rcx]
    test    dil, 1
    jz      .lower_ref_even
    and     ecx, 0x0F
    jmp     .lower_ref_done

.lower_ref_even:
    shr     ecx, 4

.lower_ref_done:
    cmp     r9d, ecx
    jne     .lower_done
    inc     rax
    inc     rdi                  
    inc     rsi
    dec     rdx
    jmp     .lower_diag

.lower_done:
    STOP_TIMER [rbp-56], OFF_LEFT_DIAG

    mov     rcx, rbx
    sub     rcx, r15
    cmp     rax, rcx
    jae     .kmer_next

    cmp     rax, [rbp-40]
    jbe     .next_shift

    mov     [rbp-40], rax

.next_shift:
    inc     r8
    jmp     .shift_loop

.shift_done:
    ; Add best match count + globalCount
    mov     r9, [rbp-40]
    add     r15, r9

    cmp     r15, rbx              ; vs KmerEnd
    jae     .kmer_next

    inc     r14                   ; Edits++
    inc     r15                   ; index++

    inc     qword [rbp-16]        ;roundsNo and IterationNo stuff
    mov     rax, [rbp-16]
    cmp     rax, [rbp-24]         ; vs IterationNo
    jg      .kmer_next

    jmp     .while_loop

.kmer_next:
    ;check EditThreshold
    cmp     r14, r10
    jg      .reject

    ; Safety counter check 
    mov     rax, [safety_counter]
    cmp     rax, [rbp-24]
    jg      .reject

    mov     r15, rbx              ; new index = old KmerEnd
    add     rbx, [rbp-8]          ; new KmerEnd = old KmerEnd + KmerSize
    cmp     rbx, r13
    cmova   rbx, r13

    cmp     r15, r13              ; if new index >= ReadLength, accept
    jae     .accept

    mov     qword [rbp-16], 1     ; roundsNo = 1
    jmp     .while_loop

.accept:
    mov     [current_position], r13
    mov     [current_edits], r14
    
    ACCUM_OUTCOME OFF_ACCEPTED
    mov     eax, 1
    jmp     .end

.reject:
    mov     [current_position], r15
    mov     [current_edits], r14
    
    ACCUM_OUTCOME OFF_REJECTED
    xor     eax, eax

.end:
    STOP_TIMER [rbp-64], OFF_TOTAL 

    vzeroupper
    add     rsp, 72             
    pop     r15
    pop     r14
    pop     r13
    pop     r12
    pop     rbx
    leave
    ret

Overwriting checkpointavxfull_test.asm


In [6]:
#[file] [threshold] [seqlen] [kmersize] [# of pairs] [debug limit]

In [21]:
!nasm -f elf64 checkpointavxfull_test.asm -o checkpointavxfull_test.o 
!gcc -c checkpointfull_test.c -o checkpointfull_test.o -mavx512f -mavx512bw 
!gcc checkpointavxfull_test.o checkpointfull_test.o -o checkpoint1_test -mavx512f -mavx512bw 
!./checkpoint1_test "../THES3/DNAPAIRS/ERR240727_1_E2_30million.txt" 0 100 100 30000 0 

Loading sequences (Fixed Len=100, KmerSize=100, Limit=30000)...

Loaded 30000 sequences total.

=== DEBUGGING FIRST 0 SEQUENCES ===

=== RUNNING FULL BENCHMARK ===

                  BENCHMARK SUMMARY               
Total Pairs              : 30000
Fixed Length             : 100 bases
Kmer Size                : 100 bases
Threshold                : 0

C Implementation:
  Accepted               : 243
  Time                   : 0.0047 seconds

ASM Implementation:
  Accepted               : 243
  Time                   : 0.0034 seconds
  Speedup vs C           : 1.39x

=== ASM INTERNAL TIMING BREAKDOWN ===
Component                       Total (s)        Calls      Avg Cycles
-------------------------------------------------------------------------------
Preprocessing (C)                0.031956        30000            2557
Total SneakySnake                0.005610        60000             224
  Main Diagonal                  0.001538        30000             123
  Right (Upper) Diag      